# Generative AI Lab Programs — Labs 6 to 10

Covers:
- **Lab 6:** Prompt Strategy Design (Zero-shot vs One-shot vs Few-shot)
- **Lab 7:** Functional Prompt Development (Summarization, Email, Content Generation)
- **Lab 8:** API Integration (OpenAI, Google Gemini, Hugging Face)
- **Lab 9:** Structured Output Generation (Python code & SQL queries)
- **Lab 10:** Strategy Evaluation (Comparative analysis of prompting methods)

**How to use:**
1. Run the Setup cells first.
2. Run each Lab's cells top to bottom.
3. For Lab 8, you only need a key for whichever API you want to try — skip the rest.


## Setup — Main LLM client (Groq)
We use **Groq** (free & fast) as the main model for Labs 6, 7, 9 and 10.
Lab 8 separately shows how to connect to OpenAI, Gemini, and Hugging Face.

Get a free Groq API key from https://console.groq.com/keys

In [ ]:
# Install the Groq Python SDK
!pip install groq -q

In [ ]:
from groq import Groq
from getpass import getpass

GROQ_API_KEY = getpass("Enter your Groq API Key: ")

client = Groq(api_key=GROQ_API_KEY)
MODEL_NAME = "llama-3.3-70b-versatile"   # change to any model available on Groq

print("Groq client ready!")

In [ ]:
# Helper function: sends a prompt to the LLM and returns the text response
def ask_llm(prompt, max_tokens=700, temperature=0.7):
    response = client.chat.completions.create(
        model=MODEL_NAME,
        max_tokens=max_tokens,
        temperature=temperature,
        messages=[
            {"role": "user", "content": prompt}
        ]
    )
    return response.choices[0].message.content

print("Helper function ready!")

---
# Lab 6: Prompt Strategy Design
**Task:** Implement and compare Zero-shot, One-shot, and Few-shot prompts for the same
job — classifying the sentiment of a customer review as Positive, Negative, or Neutral.

In [ ]:
review_to_classify = "The product arrived a day late, but the quality completely made up for it. I'm impressed!"

print("Review to classify:\n", review_to_classify)

In [ ]:
# ---- Zero-shot Prompt ----
zero_shot_prompt_6 = f"""Classify the sentiment of the following review as Positive,
Negative, or Neutral. Reply with only one word.

Review: "{review_to_classify}"
Sentiment:"""

result_zero_6 = ask_llm(zero_shot_prompt_6, max_tokens=10)
print("ZERO-SHOT OUTPUT:", result_zero_6)

In [ ]:
# ---- One-shot Prompt ----
one_shot_prompt_6 = f"""Classify the sentiment of a review as Positive, Negative, or
Neutral. Reply with only one word.

Review: "The battery died within a day and customer support was unhelpful."
Sentiment: Negative

Review: "{review_to_classify}"
Sentiment:"""

result_one_6 = ask_llm(one_shot_prompt_6, max_tokens=10)
print("ONE-SHOT OUTPUT:", result_one_6)

In [ ]:
# ---- Few-shot Prompt ----
few_shot_prompt_6 = f"""Classify the sentiment of a review as Positive, Negative, or
Neutral. Reply with only one word.

Review: "The battery died within a day and customer support was unhelpful."
Sentiment: Negative

Review: "It's an average product, does exactly what it says, nothing more."
Sentiment: Neutral

Review: "Absolutely love this! Best purchase I've made all year."
Sentiment: Positive

Review: "{review_to_classify}"
Sentiment:"""

result_few_6 = ask_llm(few_shot_prompt_6, max_tokens=10)
print("FEW-SHOT OUTPUT:", result_few_6)

### Compare effectiveness of the three strategies
The cell below asks the LLM to judge which strategy produced the most reliable answer
and why (based on prompt clarity, guidance given, and output consistency).

In [ ]:
comparison_prompt_6 = f"""A sentiment classification task was performed using three
prompting strategies on the same review. Compare their effectiveness.

Review: "{review_to_classify}"

Zero-shot prediction: {result_zero_6}
One-shot prediction: {result_one_6}
Few-shot prediction: {result_few_6}

Explain in a short table which strategy is likely to be most RELIABLE in general and why,
considering: amount of guidance given to the model, ambiguity handling, and consistency
of output format. End with a one-line recommendation."""

print(ask_llm(comparison_prompt_6))

---
# Lab 7: Functional Prompt Development
Designing **specialized, reusable prompt templates** (as Python functions) for three
different real-world functions: summarization, email creation, and content generation.

### 7.1 Summarization Function

In [ ]:
def summarize_text(text, word_limit=50):
    prompt = f"""You are an expert summarizer. Summarize the following text in
exactly {word_limit} words. Keep the key facts, remove filler, and keep it easy to read.

Text:
{text}

Summary ({word_limit} words):"""
    return ask_llm(prompt, max_tokens=200)

sample_text = """
Cloud computing allows businesses to access computing resources such as servers,
storage, and databases over the internet instead of owning physical infrastructure.
This reduces upfront hardware costs and allows companies to scale resources up or
down based on demand. Major providers like AWS, Microsoft Azure, and Google Cloud
offer a range of services including virtual machines, managed databases, and AI tools.
Cloud computing has become essential for startups and enterprises alike, enabling
faster innovation and global reach without heavy infrastructure investment.
"""

print(summarize_text(sample_text, word_limit=40))

### 7.2 Email Creation Function

In [ ]:
def generate_email(purpose, recipient_role, tone="professional", key_points=None):
    points_text = ""
    if key_points:
        points_text = "Include these key points:\n- " + "\n- ".join(key_points)

    prompt = f"""Write a {tone} email to a {recipient_role}.
Purpose of the email: {purpose}
{points_text}

Format it with a Subject line, greeting, body, and a polite closing."""
    return ask_llm(prompt, max_tokens=400)

email_output = generate_email(
    purpose="requesting a deadline extension for a project submission",
    recipient_role="project manager",
    tone="professional and polite",
    key_points=["Need 3 extra days", "Cause: unexpected technical issues", "Assure quality won't be compromised"]
)

print(email_output)

### 7.3 Content Generation Function

In [ ]:
def generate_content(content_type, topic, tone="engaging", length="short"):
    prompt = f"""Generate a {length} {content_type} about "{topic}".
Tone: {tone}.
Make it well-structured, catchy, and appropriate for the given content type."""
    return ask_llm(prompt, max_tokens=400)

content_output = generate_content(
    content_type="Instagram caption",
    topic="a new AI-powered fitness app launch",
    tone="fun and energetic",
    length="short"
)

print(content_output)

---
# Lab 8: API Integration
Connecting the same task ("Explain what an API is, in 2 lines") to **three different
LLM providers**: OpenAI, Google Gemini, and Hugging Face Inference API.

You only need a key for the provider(s) you want to try. Leave the input blank and
press Enter to skip a provider.

### 8.1 OpenAI API

In [ ]:
!pip install openai -q

In [ ]:
from openai import OpenAI

openai_key = getpass("Enter your OpenAI API Key (leave blank to skip): ")

if openai_key.strip():
    openai_client = OpenAI(api_key=openai_key)
    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": "Explain what an API is, in 2 lines."}]
    )
    print("OPENAI OUTPUT:\n")
    print(response.choices[0].message.content)
else:
    print("Skipped OpenAI (no key entered).")

### 8.2 Google Gemini API

In [ ]:
!pip install google-generativeai -q

In [ ]:
import google.generativeai as genai

gemini_key = getpass("Enter your Google Gemini API Key (leave blank to skip): ")

if gemini_key.strip():
    genai.configure(api_key=gemini_key)
    gemini_model = genai.GenerativeModel("gemini-1.5-flash")
    response = gemini_model.generate_content("Explain what an API is, in 2 lines.")
    print("GEMINI OUTPUT:\n")
    print(response.text)
else:
    print("Skipped Gemini (no key entered).")

### 8.3 Hugging Face Inference API

In [ ]:
!pip install huggingface_hub -q

In [ ]:
from huggingface_hub import InferenceClient

hf_key = getpass("Enter your Hugging Face API Token (leave blank to skip): ")

if hf_key.strip():
    hf_client = InferenceClient(token=hf_key)
    response = hf_client.chat_completion(
        model="meta-llama/Llama-3.1-8B-Instruct",
        messages=[{"role": "user", "content": "Explain what an API is, in 2 lines."}],
        max_tokens=100
    )
    print("HUGGING FACE OUTPUT:\n")
    print(response.choices[0].message.content)
else:
    print("Skipped Hugging Face (no key entered).")

---
# Lab 9: Structured Output Generation
Using structured prompting techniques to make the LLM return **valid, directly usable**
Python code and SQL queries (not just free-form text).

### 9.1 Generating valid Python code

In [ ]:
import re

def generate_python_code(task_description):
    prompt = f"""You are a Python code generator.
Task: {task_description}

Rules:
- Output ONLY valid Python code, nothing else.
- Do not include explanations or markdown formatting like ```python.
- Include a docstring describing what the function does.
- Include a small example usage with a print statement at the end.
"""
    code_output = ask_llm(prompt, max_tokens=400, temperature=0.2)

    # Clean up in case the model still wraps it in markdown fences
    code_output = re.sub(r"^```python|```$", "", code_output.strip(), flags=re.MULTILINE).strip()
    return code_output

python_task = "Write a function that checks whether a given number is a prime number."
generated_code = generate_python_code(python_task)

print("GENERATED PYTHON CODE:\n")
print(generated_code)

In [ ]:
# Validate that the generated code is syntactically correct, then run it
import ast

try:
    ast.parse(generated_code)
    print("Syntax check passed. Running the code:\n")
    exec(generated_code)
except SyntaxError as e:
    print("Syntax check FAILED:", e)

### 9.2 Generating valid SQL queries

In [ ]:
def generate_sql_query(task_description, schema):
    prompt = f"""You are a SQL query generator.

Database schema:
{schema}

Task: {task_description}

Rules:
- Output ONLY the SQL query, nothing else.
- Do not include explanations or markdown formatting like ```sql.
- Use proper SQL syntax and correct column/table names from the schema.
"""
    sql_output = ask_llm(prompt, max_tokens=200, temperature=0.2)
    sql_output = re.sub(r"^```sql|```$", "", sql_output.strip(), flags=re.MULTILINE).strip()
    return sql_output

schema = """
Table: employees (id INT, name VARCHAR, department VARCHAR, salary INT, join_date DATE)
"""

sql_task = "Find the names and salaries of employees in the 'Engineering' department who earn more than 60000, sorted by salary descending."
generated_sql = generate_sql_query(sql_task, schema)

print("GENERATED SQL QUERY:\n")
print(generated_sql)

In [ ]:
# Basic validation: check the query against the live schema using SQLite
import sqlite3

conn = sqlite3.connect(":memory:")
cur = conn.cursor()
cur.execute("""CREATE TABLE employees (
    id INTEGER, name TEXT, department TEXT, salary INTEGER, join_date TEXT
)""")
cur.executemany(
    "INSERT INTO employees VALUES (?, ?, ?, ?, ?)",
    [
        (1, "Asha", "Engineering", 75000, "2022-01-10"),
        (2, "Ravi", "Engineering", 55000, "2021-05-14"),
        (3, "Meera", "Marketing", 62000, "2020-11-01"),
        (4, "Karan", "Engineering", 68000, "2023-03-22"),
    ]
)
conn.commit()

try:
    cur.execute(generated_sql)
    rows = cur.fetchall()
    print("Query ran successfully! Results:\n")
    for row in rows:
        print(row)
except sqlite3.Error as e:
    print("SQL execution FAILED:", e)

conn.close()

---
# Lab 10: Strategy Evaluation
Comparative analysis of LLM responses across **four different prompting methodologies**
for the same task: Zero-shot, Few-shot, Chain-of-Thought, and Role-based prompting.

**Task used:** Solve a simple word problem that needs reasoning.

In [ ]:
word_problem = "A shop had 120 apples. It sold 35% of them in the morning and 40 more in the afternoon. How many apples are left?"

print("Problem:\n", word_problem)

In [ ]:
# ---- Zero-shot Prompt ----
zero_shot_prompt_10 = f"""Solve this problem and give only the final numeric answer:

{word_problem}"""

out_zero_10 = ask_llm(zero_shot_prompt_10, max_tokens=100)
print("ZERO-SHOT OUTPUT:\n", out_zero_10)

In [ ]:
# ---- Few-shot Prompt ----
few_shot_prompt_10 = f"""Solve the math word problem and give only the final numeric answer.

Problem: A basket has 80 mangoes. 25% are sold and 10 more are given away. How many are left?
Answer: 50

Problem: A store has 200 shirts. It sells 30% of them and restocks 20 more. How many shirts does it have now?
Answer: 160

Problem: {word_problem}
Answer:"""

out_few_10 = ask_llm(few_shot_prompt_10, max_tokens=100)
print("FEW-SHOT OUTPUT:\n", out_few_10)

In [ ]:
# ---- Chain-of-Thought Prompt ----
cot_prompt_10 = f"""Solve the following problem. Think through it step by step
before giving the final answer.

Problem: {word_problem}

Let's think step by step:"""

out_cot_10 = ask_llm(cot_prompt_10, max_tokens=300)
print("CHAIN-OF-THOUGHT OUTPUT:\n", out_cot_10)

In [ ]:
# ---- Role-based Prompt ----
role_prompt_10 = f"""You are a meticulous math teacher who always double-checks
calculations before answering. Solve the following problem for a student, showing
your verification step, then give the final answer clearly.

Problem: {word_problem}"""

out_role_10 = ask_llm(role_prompt_10, max_tokens=300)
print("ROLE-BASED OUTPUT:\n", out_role_10)

### Comparative Analysis
The cell below asks the LLM to evaluate all four outputs against the correct answer
on **Accuracy, Reasoning Clarity, and Conciseness**, presented as a table.

In [ ]:
evaluation_prompt_10 = f"""Evaluate four LLM responses to the same math word problem
across different prompting methodologies. For each, rate Accuracy (is the final number
correct?), Reasoning Clarity (out of 5), and Conciseness (out of 5). Present results as
a markdown table with columns: Method, Final Answer, Accuracy, Reasoning Clarity,
Conciseness. End with a one-line conclusion on which methodology worked best for this
type of task and why.

Problem: {word_problem}

Zero-shot response: {out_zero_10}

Few-shot response: {out_few_10}

Chain-of-Thought response: {out_cot_10}

Role-based response: {out_role_10}
"""

print(ask_llm(evaluation_prompt_10, max_tokens=600))

---
## End of Notebook
Labs 6 to 10 are complete:
- Lab 6 compared Zero/One/Few-shot prompting for sentiment classification.
- Lab 7 built reusable prompt functions for summarization, email writing, and content generation.
- Lab 8 connected to OpenAI, Google Gemini, and Hugging Face APIs.
- Lab 9 generated and validated structured Python code and SQL queries.
- Lab 10 compared Zero-shot, Few-shot, Chain-of-Thought, and Role-based prompting strategies.
